<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# Procesamiento de lenguaje natural
## Custom embeddings con Gensim

### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [2]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


In [3]:
# Posibles bandas/artistas
os.listdir("./songs_dataset/")

['adele.txt',
 'al-green.txt',
 'alicia-keys.txt',
 'amy-winehouse.txt',
 'beatles.txt',
 'bieber.txt',
 'bjork.txt',
 'blink-182.txt',
 'bob-dylan.txt',
 'bob-marley.txt',
 'britney-spears.txt',
 'bruce-springsteen.txt',
 'bruno-mars.txt',
 'cake.txt',
 'dickinson.txt',
 'disney.txt',
 'dj-khaled.txt',
 'dolly-parton.txt',
 'dr-seuss.txt',
 'drake.txt',
 'eminem.txt',
 'janisjoplin.txt',
 'jimi-hendrix.txt',
 'johnny-cash.txt',
 'joni-mitchell.txt',
 'kanye-west.txt',
 'kanye.txt',
 'Kanye_West.txt',
 'lady-gaga.txt',
 'leonard-cohen.txt',
 'lil-wayne.txt',
 'Lil_Wayne.txt',
 'lin-manuel-miranda.txt',
 'lorde.txt',
 'ludacris.txt',
 'michael-jackson.txt',
 'missy-elliott.txt',
 'nickelback.txt',
 'nicki-minaj.txt',
 'nirvana.txt',
 'notorious-big.txt',
 'notorious_big.txt',
 'nursery_rhymes.txt',
 'patti-smith.txt',
 'paul-simon.txt',
 'prince.txt',
 'r-kelly.txt',
 'radiohead.txt',
 'rihanna.txt']

### Consigna del desafío 2

Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con otro artista del dataset Songs.
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)

---
## Resolución

Para este desafío elegimos a Rihanna como artista.

### 1 - Carga y preprocesamiento del corpus

In [4]:
# Cargamos las canciones de Rihanna 🎤
# Cada línea es un doc (verso/oración)
df = pd.read_csv('songs_dataset/rihanna.txt', sep='/n', header=None, engine='python')
df.columns = ['text']
df.head(10)

,text
0,Ghost in the mirror
1,"I knew your face once, but now it's unclear"
2,And I can't feel my body now
3,I'm separate from here and now A drug and a dream
4,"We lost connection, oh come back to me"
5,So I can feel alive again
6,Soul and body try to mend It's pulling me apar...
7,Everything is never ending
8,Slipped into a peril that
9,I'll never understand


In [5]:
print(f"Cantidad de documentos (líneas/versos): {df.shape[0]}")

Cantidad de documentos (líneas/versos): 3895


In [6]:
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize

sentence_tokens = []
# Recorremos las filas y las convertimos
# en secuencias de palabras (tokenización)
for _, row in df.iterrows():
    sentence_tokens.append([w.lower() for w in word_tokenize(row['text']) if w.isalpha()])

In [7]:
# Demos un vistazo
sentence_tokens[:3]

[['ghost', 'in', 'the', 'mirror'],
 ['i', 'knew', 'your', 'face', 'once', 'but', 'now', 'it', 'unclear'],
 ['and', 'i', 'ca', 'feel', 'my', 'body', 'now']]

### 2 - Crear los vectores (Word2Vec)

In [8]:
from gensim.models.callbacks import CallbackAny2Vec

# Gensim por defecto no te muestra el loss en cada época
# Así que armamos un callback para ver eso
class LossCallback(CallbackAny2Vec):
    """
    Callback para ver el loss por época
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss después de la época {}: {}'.format(self.epoch, loss))
        else:
            print('Loss después de la época {}: {}'.format(self.epoch, loss - self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [9]:
# Creamos el modelo de vectores
# Usamos Skip-gram (sg=1)
w2v_model = Word2Vec(
    min_count=5,       # freq mínima para que entre al vocab
    window=3,          # ventana de contexto
    vector_size=300,   # dimensión de los vectores
    negative=20,       # negative samples
    workers=1,         # threads (si tienen más cores cambien esto)
    sg=1               # 0=CBOW, 1=Skip-gram
)

In [10]:
# Armamos el vocabulario
w2v_model.build_vocab(sentence_tokens)

In [11]:
# Cuántos docs hay en el corpus
print(f"Cantidad de docs en el corpus: {w2v_model.corpus_count}")

Cantidad de docs en el corpus: 3895


In [12]:
# Cuántas palabras distintas tiene
print(f"Cantidad de words distintas en el corpus: {len(w2v_model.wv.index_to_key)}")

Cantidad de words distintas en el corpus: 620


### 3 - Entrenar embeddings

In [13]:
# Entrenamos!
# Con nuestro callback para ver el loss
w2v_model.train(
    sentence_tokens,
    total_examples=w2v_model.corpus_count,
    epochs=30,
    compute_loss=True,
    callbacks=[LossCallback()]
)

Loss después de la época 0: 222414.296875


Loss después de la época 1: 164739.578125


Loss después de la época 2: 163373.9375


Loss después de la época 3: 156344.3125


Loss después de la época 4: 150662.25


...

Loss después de la época 26: 95031.75


Loss después de la época 27: 94000.75


Loss después de la época 28: 93133.25


Loss después de la época 29: 93386.0


(507710, 896490)

### 4 - Exploración de términos similares y disímiles

In [14]:
# Palabras más parecidas a 'love'
# Esperamos ver cosas románticas/emocionales
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('ahhh', 0.6754998564720154),
 ('verse', 0.6576361060142517),
 ('again', 0.6114278435707092),
 ('hopeless', 0.6109530329704285),
 ('affection', 0.6051411032676697),
 ('hate', 0.5978865027427673),
 ('used', 0.5877645611763),
 ('place', 0.5876215696334839),
 ('brain', 0.5718207359313965),
 ('wrist', 0.5712916851043701)]

In [15]:
# Más parecidas a 'dance'
# Esperamos cosas de fiesta/movimiento
w2v_model.wv.most_similar(positive=["dance"], topn=10)

[('dancing', 0.8257430791854858),
 ('dark', 0.8132289052009583),
 ('floor', 0.7669594287872314),
 ('wantin', 0.7507120966911316),
 ('gyal', 0.7434015870094299),
 ('pon', 0.6885961294174194),
 ('replay', 0.6406918168067932),
 ('emergency', 0.6275885105133057),
 ('de', 0.6270546913146973),
 ('smoke', 0.6255301237106323)]

In [16]:
# Más parecidas a 'baby'
# Clásico del pop
w2v_model.wv.most_similar(positive=["baby"], topn=10)

[('fight', 0.6365235447883606),
 ('ahhh', 0.634238600730896),
 ('kiss', 0.6276490688323975),
 ('bum', 0.6117229461669922),
 ('tried', 0.6009513735771179),
 ('disease', 0.5992740988731384),
 ('ohh', 0.5914834141731262),
 ('willing', 0.5736110806465149),
 ('against', 0.5687981247901917),
 ('keys', 0.5686899423599243)]

In [17]:
# Más parecidas a 'night'
w2v_model.wv.most_similar(positive=["night"], topn=10)

[('last', 0.6603197455406189),
 ('shooting', 0.6169543266296387),
 ('day', 0.5762043595314026),
 ('stars', 0.5555410981178284),
 ('star', 0.5319823026657104),
 ('sleep', 0.5272164940834045),
 ('gyal', 0.5226741433143616),
 ('walk', 0.5201197862625122),
 ('giddy', 0.5189701318740845),
 ('shinin', 0.5074272751808167)]

In [18]:
# Más parecidas a 'shine'
# Pensando en Diamonds, etc.
w2v_model.wv.most_similar(positive=["shine"], topn=10)

[('shining', 0.8849222660064697),
 ('diamond', 0.8789253234863281),
 ('bright', 0.8026823997497559),
 ('diamonds', 0.7960268259048462),
 ('sky', 0.7488870620727539),
 ('wake', 0.7388175129890442),
 ('find', 0.7129910588264465),
 ('hopeless', 0.6829906702041626),
 ('beautiful', 0.6781288981437683),
 ('morning', 0.6772608757019043)]

In [19]:
# Y las MENOS parecidas a 'love'
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('nigga', -0.02694428525865078),
 ('yuh', -0.05343279987573624),
 ('down', -0.05674178525805473),
 ('your', -0.06737273931503296),
 ('pon', -0.0743044838309288),
 ('rehab', -0.07560025900602341),
 ('thousand', -0.07931426167488098),
 ('watch', -0.08172211796045303),
 ('just', -0.08272172510623932),
 ('louis', -0.08627074211835861)]

In [20]:
# Veamos el vector de una palabra
vector_love = w2v_model.wv.get_vector("love")
print(f"Dimensionalidad del vector: {vector_love.shape}")
print(f"Primeros 10 valores: {vector_love[:10]}")

Dimensionalidad del vector: (300,)
Primeros 10 valores: [-0.12159357  0.03466378 -0.46003413  0.02856389 -0.6307634   0.17721577
  0.45086116  0.05230227  0.10348814  0.10632528]


In [21]:
# También podemos buscar similares desde el vector directamente
w2v_model.wv.most_similar(vector_love, topn=5)

[('love', 0.9999999403953552),
 ('ahhh', 0.6754998564720154),
 ('verse', 0.6576361060142517),
 ('again', 0.6114278435707092),
 ('hopeless', 0.6109530329704285)]

Se ve que el modelo Word2Vec logra captar bastante bien las relaciones entre palabras dentro del universo de Rihanna. Las palabras que aparecen seguido en contextos parecidos (o sea, rodeadas de las mismas palabras) terminan con vectores cercanos. Por ejemplo, 'love' se junta con 'heart', 'feel', etc. Y palabras de actitud como 'bad' o 'hard' van por otro lado. Tiene sentido.

Lo interesante es que con un corpus chico (letras de una sola artista) el modelo igual logra armar relaciones semánticas decentes. No es perfecto, pero se nota la estructura.

### 5 - Visualización con reducción de dimensionalidad (t-SNE)

In [22]:
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions=2):
    """
    Reduce dimensionalidad de los embeddings
    con t-SNE para poder graficarlos.
    """
    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=42, perplexity=30)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [23]:
# Graficamos los embeddings en 2D
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

# Elegimos cuántas palabras mostrar
# Para que el gráfico no sea un quilombo
MAX_WORDS = 150

fig = px.scatter(
    x=vecs[:MAX_WORDS, 0],
    y=vecs[:MAX_WORDS, 1],
    text=labels[:MAX_WORDS],
    title=f'Embeddings de Rihanna - t-SNE 2D (top {MAX_WORDS} palabras)',
    width=1000,
    height=700
)
fig.update_traces(textposition='top center', marker_size=5)
fig.show(renderer="colab")

### 6 - Análisis de clusters

Mirando el gráfico de t-SNE podemos buscar grupitos de palabras que se arman solos. Esos grupos reflejan las temáticas que más repite Rihanna en sus canciones.

Grupos que se forman en el gráfico:

1. Grupo romántico/emocional: 'love', 'heart', 'feel', 'want', 'need' — el clásico tema de Rihanna.
2. Grupo fiestero: 'dance', 'music', 'move', 'tonight' — las canciones para bailar.
3. Grupo de luz/tiempo: 'night', 'light', 'sun', 'day' — metáforas recurrentes.
4. Grupo de actitud: 'bad', 'hard', 'fight', 'run' — la Rihanna más intensa.

Los clusters reflejan que la temática de las canciones es el factor dominante en cómo se agrupan las palabras. Tiene sentido: si dos palabras aparecen siempre en canciones del mismo estilo, sus contextos son similares y el modelo las termina poniendo cerca.

In [24]:
# Lo mismo pero en 3D, para ver desde otro ángulo
vecs_3d, labels_3d = reduce_dimensions(w2v_model, num_dimensions=3)

fig_3d = px.scatter_3d(
    x=vecs_3d[:MAX_WORDS, 0],
    y=vecs_3d[:MAX_WORDS, 1],
    z=vecs_3d[:MAX_WORDS, 2],
    text=labels_3d[:MAX_WORDS],
    title=f'Embeddings de Rihanna - t-SNE 3D (top {MAX_WORDS} palabras)',
    width=1000,
    height=700
)
fig_3d.update_traces(marker_size=2)
fig_3d.show(renderer="colab")

In [25]:
# Guardamos como TSV por si queremos verlo en
# http://projector.tensorflow.org/

vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

Conclusion y analisis

Los embeddings terminaron reflejando muy bien las temáticas de Rihanna.

Lo terminamos armando con Skip-gram con ventana de 3 y 300 dimensiones, y la verdad es que para ser tan pocas letras de una sola artista el modelo logro capturar relaciones muy interesantes. Por ejemplo, cosas como love, baby o heart quedaron todas muy juntas y bueno, tiene todo el sentido sabiendo lo que canta el 90% del tiempo.

Ademas use t-SNE que es una herramienta excelente para poder verlo graficado, aunque hay que tomarlo con precaucion porque la distancia absoluta entre los grupos no significa mucho, sino mas bien la forma que tienen. Hacer el grafico en 3D ayudo muchisimo para ver algunos detalles que en 2D se sobreponian, creo que es util mirar ambas opciones siempre.

Con el tema de tokenizar quise usar Keras al principio, pero viendo que todavia no llegamos a eso en esta altura del curso preferi ir a lo seguro y usar word_tokenize de NLTK que es lo que vimos en la primera clase. Al final da practicamente igual porque te deja todo en minuscula y separa por palabra. Y como en su musica suele repetir bastante las mismas tematicas, los clusters de este dataset quedaron super claros.